In [ ]:
from google import genai
from google.genai import types
import gradio as gr
from pydantic import BaseModel
from typing import Literal, AsyncGenerator
import os

In [ ]:
# check if api key is present
assert os.getenv('GOOGLE_API_KEY')

# Prompt Engineering

In [21]:
client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))

SYS_PROMPT = """
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""

chat = client.chats.create(model="gemini-2.0-flash",
                           config=types.GenerateContentConfig(
                               system_instruction=SYS_PROMPT,
                               temperature=0.2
                           ))

# Chatbot interface

In [23]:
def answer(msg: str, hist: list) -> str:
    if not msg.strip():
        return "you can't send an empty message"
    try:
        response = chat.send_message(msg)
        return response.text
    except Exception as e:
        print(f"Error: {str(e)}")
        return "Internal error. Try again later "

async def stream_answer(msg: str, hist: list) -> AsyncGenerator[str, None]:
    if not msg.strip():
        yield "you can't send an empty message"
    else:
        try:
            buffer = ""
            response = chat.send_message_stream(msg)
            for chunk in response:
                buffer += chunk.text
                yield buffer
        except Exception as e:
            print(f"Error: {str(e)}")
            yield "Internal error. Try again later "
            return

In [ ]:
demo = gr.ChatInterface(stream_answer,
                 type="messages",
                 title="OrderBot",
                 description="Hello I am OrderBot, an automated service that collect orders for our pizza restaurant",
                 theme="soft",)
try:
    demo.launch()
except Exception as e:
    demo.close()

# Order Summary

In [18]:
class BaseItem(BaseModel):
    type: str
    price: float

class ItemWithSize(BaseItem):
    size: Literal['Small', 'Medium', 'Large', 'One Size']

class Summary(BaseModel):
    pizza: list[ItemWithSize]
    toppings: list[BaseItem]
    drinks: list[ItemWithSize]
    sides: list[ItemWithSize]
    total_price: float


In [19]:
summary_prompt = "create a json summary of the previous food order. Itemize the price for each item"
response = chat.send_message(summary_prompt, config={
        'response_mime_type': 'application/json',
        'response_schema': Summary,
    })
print(response.text)

{
  "pizza": [
    {
      "type": "pepperoni",
      "price": 17.50,
      "size": "Medium"
    }
  ],
  "toppings": [
    {
      "type": "mushrooms",
      "price": 1.50
    }
  ],
  "drinks": [],
  "sides": [
    {
      "type": "fries",
      "price": 3.50,
      "size": "Small"
    }
  ],
  "total_price": 22.50
}
